# Decision Tree baseline — Handwritten Digits

Notebook cơ bản cho bộ `sklearn.datasets.load_digits`: 1.797 ảnh 8×8, 64 pixel features và 10 lớp. Notebook có thể upload trực tiếp lên Kaggle.

**Kaggle accelerator:** chọn **None (CPU)**. `DecisionTreeClassifier` của scikit-learn không dùng GPU, và dataset này rất nhỏ.

## 1. Import và cấu hình

In [ ]:
import json
import platform
import time
from datetime import UTC, datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.datasets import load_digits
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPERIMENT_ID = "dt_digits_baseline"
TARGET = "target"
FEATURES = [f"pixel_{index:02d}" for index in range(64)]

IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
FIGURES_DIR = OUTPUT_ROOT / "figures"
RESULTS_DIR = OUTPUT_ROOT / "results"
MODELS_DIR = OUTPUT_ROOT / "models"
for directory in (FIGURES_DIR, RESULTS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
print({"python": platform.python_version(), "sklearn": sklearn.__version__, "kaggle": IS_KAGGLE})

## 2. Đọc dữ liệu

Notebook ưu tiên `digits.csv` được attach trong Kaggle Input. Nếu không có, nó dùng bản tích hợp sẵn trong scikit-learn nên không cần Internet.

In [ ]:
REQUIRED_COLUMNS = set(FEATURES + [TARGET])
SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent]
if Path("/kaggle/input").exists():
    SEARCH_ROOTS.insert(0, Path("/kaggle/input"))

def csv_has_columns(path):
    try:
        return REQUIRED_COLUMNS.issubset(pd.read_csv(path, nrows=1).columns)
    except (OSError, UnicodeError, ValueError, pd.errors.ParserError):
        return False

def find_digits_csv():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("digits.csv"):
            if csv_has_columns(path):
                return path
    return None

digits_path = find_digits_csv()
if digits_path is not None:
    digits_df = pd.read_csv(digits_path)[FEATURES + [TARGET]]
    data_source = str(digits_path)
else:
    digits = load_digits()
    digits_df = pd.DataFrame(digits.data, columns=FEATURES)
    digits_df[TARGET] = digits.target
    data_source = "sklearn.datasets.load_digits()"

assert digits_df.shape[1] == 65
assert digits_df[TARGET].nunique() == 10
assert not digits_df.isna().any().any()
assert digits_df[FEATURES].to_numpy().min() >= 0
assert digits_df[FEATURES].to_numpy().max() <= 16

print("Nguồn:", data_source)
print("Kích thước:", digits_df.shape)
display(digits_df.head())

## 3. Representation và phân bố lớp

Mỗi mẫu là một ảnh grayscale 8×8 đã được flatten thành 64 cột; giá trị pixel nằm trong [0, 16]. Khác với 16 đặc trưng hình học được thiết kế sẵn của Letter Recognition, đây là raw-pixel representation.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for label, ax in enumerate(axes.flat):
    sample = digits_df.loc[digits_df[TARGET] == label, FEATURES].iloc[0].to_numpy().reshape(8, 8)
    ax.imshow(sample, cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"Label {label}")
    ax.axis("off")
fig.suptitle("Handwritten Digits - one sample per class")
fig.tight_layout()
plt.show()

class_counts = digits_df[TARGET].value_counts().sort_index()
display(class_counts.rename("count").to_frame().T)
print("Class imbalance ratio:", round(class_counts.max() / class_counts.min(), 3))

## 4. Chia train/test và huấn luyện baseline

Split được tạo đúng một lần với stratification. Không scale pixel vì cây quyết định không cần chuẩn hóa khoảng giá trị.

In [ ]:
X = digits_df[FEATURES]
y = digits_df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train/Test:", X_train.shape, X_test.shape)

model = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
started = time.perf_counter()
model.fit(X_train, y_train)
training_seconds = time.perf_counter() - started

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
metrics = {
    "train_accuracy": accuracy_score(y_train, y_train_pred),
    "test_accuracy": accuracy_score(y_test, y_test_pred),
    "error_rate": 1 - accuracy_score(y_test, y_test_pred),
    "precision_macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
    "training_seconds": training_seconds,
    "tree_depth": model.get_depth(),
    "leaf_count": model.get_n_leaves(),
}
display(pd.Series(metrics, name="value").to_frame().round(4))
display(pd.DataFrame(classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)).T.round(3))

## 5. Confusion matrix và các lỗi phân loại

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, labels=model.classes_, cmap="Blues", colorbar=False, ax=ax
)
ax.set_title("Handwritten Digits - Decision Tree baseline")
fig.tight_layout()
confusion_path = FIGURES_DIR / f"{EXPERIMENT_ID}__confusion_matrix.png"
fig.savefig(confusion_path, dpi=200, bbox_inches="tight")
plt.show()

wrong_positions = np.flatnonzero(y_test.to_numpy() != y_test_pred)[:15]
fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, position in zip(axes.flat, wrong_positions):
    image = X_test.iloc[position].to_numpy().reshape(8, 8)
    ax.imshow(image, cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"True {y_test.iloc[position]} / Pred {y_test_pred[position]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Misclassified examples")
fig.tight_layout()
mistakes_path = FIGURES_DIR / f"{EXPERIMENT_ID}__misclassified_examples.png"
fig.savefig(mistakes_path, dpi=200, bbox_inches="tight")
plt.show()

## 6. Feature importance theo vị trí pixel

In [ ]:
importance_map = model.feature_importances_.reshape(8, 8)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(importance_map, cmap="mako", square=True, ax=ax)
ax.set_title("Decision Tree - Gini importance by pixel")
ax.set(xlabel="Pixel column", ylabel="Pixel row")
fig.tight_layout()
importance_path = FIGURES_DIR / f"{EXPERIMENT_ID}__feature_importance.png"
fig.savefig(importance_path, dpi=200, bbox_inches="tight")
plt.show()

top_pixels = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10)
display(top_pixels.rename("importance").to_frame())

## 7. Trực quan ba tầng đầu của cây

In [ ]:
fig, ax = plt.subplots(figsize=(24, 11))
plot_tree(
    model, feature_names=FEATURES, class_names=[str(value) for value in model.classes_],
    max_depth=3, filled=True, rounded=True, fontsize=7, ax=ax
)
ax.set_title("Handwritten Digits - first three levels of the baseline tree")
fig.tight_layout()
tree_path = FIGURES_DIR / f"{EXPERIMENT_ID}__tree_top_levels.png"
fig.savefig(tree_path, dpi=200, bbox_inches="tight")
plt.show()

## 8. Lưu model và result contract

In [ ]:
model_path = MODELS_DIR / f"{EXPERIMENT_ID}.joblib"
joblib.dump(model, model_path)

result = {
    "schema_version": "1.0",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "handwritten_digits",
    "model": "DecisionTreeClassifier",
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": True},
    "metrics": {key: float(value) if isinstance(value, (float, np.floating)) else int(value) for key, value in metrics.items()},
    "artifacts": {
        "figure_paths": [str(confusion_path), str(mistakes_path), str(importance_path), str(tree_path)],
        "model_path": str(model_path),
    },
    "notes": "CPU baseline; 8x8 raw pixels; criterion=gini; no scaling, pruning, or tuning.",
    "created_at_utc": datetime.now(UTC).isoformat(),
}
result_path = RESULTS_DIR / f"{EXPERIMENT_ID}.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("Model:", model_path)
print("Result:", result_path)
print("Figures:", *result["artifacts"]["figure_paths"], sep="\n- ")

## Kết luận baseline

Raw pixels giúp notebook đơn giản nhưng cây không tận dụng cấu trúc không gian của ảnh. Hãy dùng kết quả này làm mốc trước khi thử pruning hoặc so sánh Random Forest/SVM/KNN trên cùng split.